# Data Cleaning and Processing: Processing denormalised data
# Mission of data engineer: to enable data analyst to get insights of the customers' behaviours on the website to sales and marketing team for a higher marketing conversion and sales rate

## Import libraries

In [2]:
import os
import pandas_gbq
from google.cloud import bigquery
import pandas as pd
import numpy as np

In [3]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS']='creds/analytics-training-485714-ccd6a252ff9d.json'

In [4]:
client = bigquery.Client()

In [5]:
G_QUERY = """
SELECT
  *
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210131`
"""

<b>pd.read_gbq is deprecated. Let's use pandas_gbq</b>

In [6]:
df = pandas_gbq.read_gbq(G_QUERY, project_id='analytics-training-485714')

Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████|


# Let's get to understand the data first

In [7]:
df.head(2)

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
0,20210131,1612069510766593,page_view,"[{'key': 'gclid', 'value': {'string_value': No...",<NA>,NaN,6595101026,<NA>,None,1026454.4271112504,...,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]
1,20210131,1612069529243877,scroll,"[{'key': 'debug_mode', 'value': {'string_value...",<NA>,NaN,9011338476,<NA>,None,1026454.4271112504,...,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]


## Data Dictionary
* event_date: The date of the event (YYYYMMDD).
* event_timestamp: The time of the event in microseconds (UTC).
* event_name: Name of the event (e.g., page_view, purchase, session_start).
* event_params: context of event
* event_previous_timestamp: Timestamp of the previous event for this user.
* event_value_in_usd: Total value of the event (usually for purchases).
* event_bundle_sequence_id: Sequential ID of the bundle the event was uploaded in. Rarely used
* event_server_timestamp_offset: Sequential ID of the bundle the event was uploaded in.
* user_id: The Signed-in User ID (if available). Mostly null for public data
* user_pseudo_id: The unauthenticated cookie ID / App Instance ID.
* privacy_info: Contains analytics_storage, ads_storage.
* user_properties: Attributes that define the user (sticky), not the event.
* user_first_touch_timestamp: Time (micros) when the user first opened the app/site.
* user_ltv: Lifetime Value struct (revenue, currency).
* device: Technical details about the user's browser/device.
* geo: Standard geographical data derived from IP address.
* app_info: App version, ID, and installer store.
* traffic_source: How the user arrived at the site for this specific event (User acquisition is stored in user_properties).
* stream_id: ID of the data stream (Web vs iOS vs Android).
* platform: 'WEB', 'IOS', or 'ANDROID'.
* event_dimensions: not found
* ecommerce: summary of event item
* items: This contains the products involved in the event. Populated mostly for view_item, add_to_cart, and purchase.

## Check data

In [8]:
df.dtypes

event_date                        object
event_timestamp                    Int64
event_name                        object
event_params                      object
event_previous_timestamp           Int64
event_value_in_usd               float64
event_bundle_sequence_id           Int64
event_server_timestamp_offset      Int64
user_id                           object
user_pseudo_id                    object
privacy_info                      object
user_properties                   object
user_first_touch_timestamp         Int64
user_ltv                          object
device                            object
geo                               object
app_info                          object
traffic_source                    object
stream_id                          Int64
platform                          object
event_dimensions                  object
ecommerce                         object
items                             object
dtype: object

In [9]:
pd.set_option('display.max_columns', None)

In [10]:
df.head()

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
0,20210131,1612069510766593,page_view,"[{'key': 'gclid', 'value': {'string_value': No...",<NA>,NaN,6595101026,<NA>,None,1026454.4271112504,"{'analytics_storage': None, 'ads_storage': Non...",[],1612069510766593,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]
1,20210131,1612069529243877,scroll,"[{'key': 'debug_mode', 'value': {'string_value...",<NA>,NaN,9011338476,<NA>,None,1026454.4271112504,"{'analytics_storage': None, 'ads_storage': Non...",[],1612069510766593,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]
2,20210131,1612069515781635,page_view,"[{'key': 'debug_mode', 'value': {'string_value...",<NA>,NaN,-6830522854,<NA>,None,1026454.4271112504,"{'analytics_storage': None, 'ads_storage': Non...",[],1612069510766593,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]
3,20210131,1612069530073506,user_engagement,"[{'key': 'page_location', 'value': {'string_va...",<NA>,NaN,-8264942910,<NA>,None,1026454.4271112504,"{'analytics_storage': None, 'ads_storage': Non...",[],1612069510766593,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]
4,20210131,1612069510766593,session_start,"[{'key': 'ga_session_number', 'value': {'strin...",<NA>,NaN,6595101026,<NA>,None,1026454.4271112504,"{'analytics_storage': None, 'ads_storage': Non...",[],1612069510766593,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",None,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenu...",[]


In [11]:
df.describe()

,event_timestamp,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_first_touch_timestamp,stream_id
count,26489.0,0.0,0.0,26489.0,0.0,25531.0,26489.0
mean,1612095087055260.5,<NA>,NaN,33156494.109215,<NA>,1611312189110824.5,2100450278.0
std,25258733143.024815,<NA>,NaN,5771147578.882031,<NA>,3721683625292.417969,0.0
min,1612051200657906.0,<NA>,NaN,-9999956402.0,<NA>,1572273976181201.0,2100450278.0
25%,1612073239248363.0,<NA>,NaN,-4903276745.0,<NA>,1612056201827953.0,2100450278.0
50%,1612094855750025.0,<NA>,NaN,39528456.0,<NA>,1612082282650821.0,2100450278.0
75%,1612116827810457.0,<NA>,NaN,4989680497.0,<NA>,1612110815163078.0,2100450278.0
max,1612137595412363.0,<NA>,NaN,9998038061.0,<NA>,1612137591044846.0,2100450278.0


As the data structure is nested and complex, pd.describe() doesn't help a lot

### Let's make a bit more sense of the data by picking a customer and see their interaction with the web

In [12]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', 100)

In [13]:
df[df.user_properties.map(len) > 0 ]

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items


In [14]:
df[df['items'].map(len) > 0 ].head(2)

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
23530,20210131,1612091514175516,view_item,"[{'key': 'engaged_session_event', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'debug_mode', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'engagement_time_msec', 'value': {'string_value': None, 'int_value': 7344, 'float_value': None, 'double_value': None}}, {'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/asearch.html', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_id', 'value': {'string_value': None, 'int_value': 5349214666, 'float_value': None, 'double_value': None}}, {'key': 'page_title', 'value': {'string_value': 'Store search results', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'session_engaged', 'value': {'string_value': '1', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_number', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'page_referrer', 'value': {'string_value': None, 'int_value': None, 'float_value': None, 'double_value': None}}]",<NA>,NaN,-5814762287,<NA>,None,1540124.2144280285,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1612091231357744,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Apple', 'mobile_model_name': 'Safari', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': None, 'operating_system': 'Web', 'operating_system_version': 'Intel 10.15', 'vendor_id': None, 'advertising_id': None, 'language': None, 'is_limited_ad_tracking': 'No', 'time_zone_offset_seconds': None, 'web_info': {'browser': 'Chrome', 'browser_version': '86.0'}}","{'continent': 'Americas', 'sub_continent': 'Northern America', 'country': 'United States', 'region': 'Oregon', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'google'}",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_in_usd': None, 'shipping_value': None, 'tax_value_in_usd': None, 'tax_value': None, 'unique_items': 1, 'transaction_id': '(not set)'}","[{'item_id': 'GGOEAFBA115599', 'item_name': 'Android Super Hero 3D Framed Art', 'item_brand': 'Android', 'item_variant': '(not set)', 'item_category': '', 'item_category2': '(not set)', 'item_category3': '(not set)', 'item_category4': '(not set)', 'item_category5': '(not set)', 'price_in_usd': None, 'price': 40.0, 'quantity': None, 'item_revenue_in_usd': None, 'item_revenue': None, 'item_refund_in_usd': None, 'item_refund': None, 'coupon': '(not set)', 'affiliation': '(not set)', 'location_id': '(not set)', 'item_list_id': '(not set)', 'item_list_name': '(not set)', 'item_list_index': '4', 'promotion_id': '(not set)', 'promotion_name': '(not set)', 'creative_name': '(not set)', 'creative_slot': '(not set)'}]"
23531,20210131,1612091260872757,add_to_cart,"[{'key': 'session_engaged', 'value': {'string_value': '1', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/asearch.html', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'page_title', 'value': {'string_value': 'Store search results', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_id', 'value': {'string_value': None, 'int_value': 5349214666, 'float_value': None, 'double_value': None

In [15]:
df.event_name.unique().tolist()

['page_view',
 'scroll',
 'user_engagement',
 'session_start',
 'first_visit',
 'view_promotion',
 'view_item',
 'view_search_results',
 'add_payment_info',
 'add_shipping_info',
 'click',
 'select_promotion',
 'add_to_cart',
 'select_item',
 'begin_checkout',
 'purchase']

In [16]:
df.platform.unique().tolist()

['WEB']

## The data is massive
below is from one customer having access within one day

In [16]:
df[df.user_pseudo_id=='1617434.1535145542']['event_date'].unique()

array(['20210131'], dtype=object)

In [17]:
df[df.user_pseudo_id=='1617434.1535145542'].shape

(55, 23)

In [18]:
df[df.user_pseudo_id=='1617434.1535145542']['event_params'].explode()

619      {'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/signin.html', 'int_value': None, 'float_value': None, 'double_value': None}}
619                       {'key': 'page_title', 'value': {'string_value': 'The Google Merchandise Store - Log In', 'int_value': None, 'float_value': None, 'double_value': None}}
619                                                             {'key': 'debug_mode', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}
619                                                {'key': 'engagement_time_msec', 'value': {'string_value': None, 'int_value': 2445, 'float_value': None, 'double_value': None}}
619                                                  {'key': 'engaged_session_event', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}
                                                                                           ...                

# Let's start cleaning and processing

## Column names: pretty standardised and clean

In [19]:
df.columns.tolist()

['event_date',
 'event_timestamp',
 'event_name',
 'event_params',
 'event_previous_timestamp',
 'event_value_in_usd',
 'event_bundle_sequence_id',
 'event_server_timestamp_offset',
 'user_id',
 'user_pseudo_id',
 'privacy_info',
 'user_properties',
 'user_first_touch_timestamp',
 'user_ltv',
 'device',
 'geo',
 'app_info',
 'traffic_source',
 'stream_id',
 'platform',
 'event_dimensions',
 'ecommerce',
 'items']

## Let's start with columns with flat data structure first

In [20]:
df.event_date

0        20210131
1        20210131
2        20210131
3        20210131
4        20210131
           ...   
26484    20210131
26485    20210131
26486    20210131
26487    20210131
26488    20210131
Name: event_date, Length: 26489, dtype: object

In [21]:
df.event_date = pd.to_datetime(df.event_date)

In [22]:
df.event_date.isna().sum()

np.int64(0)

In [23]:
df.event_timestamp = pd.to_datetime(df.event_timestamp, unit='us')
df.event_timestamp

0       2021-01-31 05:05:10.766593
1       2021-01-31 05:05:29.243877
2       2021-01-31 05:05:15.781635
3       2021-01-31 05:05:30.073506
4       2021-01-31 05:05:10.766593
                   ...            
26484   2021-01-31 04:45:27.961419
26485   2021-01-31 04:45:35.640396
26486   2021-01-31 04:55:08.660405
26487   2021-01-31 04:46:01.050112
26488   2021-01-31 06:35:49.444471
Name: event_timestamp, Length: 26489, dtype: datetime64[us]

In [24]:
df.event_timestamp.isna().sum()

np.int64(0)

In [25]:
df.event_name.unique()

array(['page_view', 'scroll', 'user_engagement', 'session_start',
       'first_visit', 'view_promotion', 'view_item',
       'view_search_results', 'add_payment_info', 'add_shipping_info',
       'click', 'select_promotion', 'add_to_cart', 'select_item',
       'begin_checkout', 'purchase'], dtype=object)

In [26]:
df.event_name.isna().sum()

np.int64(0)

In [27]:
df.event_previous_timestamp

0        <NA>
1        <NA>
2        <NA>
3        <NA>
4        <NA>
         ... 
26484    <NA>
26485    <NA>
26486    <NA>
26487    <NA>
26488    <NA>
Name: event_previous_timestamp, Length: 26489, dtype: Int64

In [28]:
df.event_previous_timestamp.isna().sum()

np.int64(26489)

In [40]:
df.event_previous_timestamp = pd.to_datetime(df.event_previous_timestamp, unit='us')

In [41]:
df.event_value_in_usd

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
26484   NaN
26485   NaN
26486   NaN
26487   NaN
26488   NaN
Name: event_value_in_usd, Length: 26489, dtype: float64

In [42]:
df.event_value_in_usd.isna().sum()

np.int64(26489)

In [43]:
df.event_bundle_sequence_id

0         6595101026
1         9011338476
2        -6830522854
3        -8264942910
4         6595101026
            ...     
26484    -2984968999
26485    -4330025487
26486     4717102046
26487    -9982755916
26488    -9352901040
Name: event_bundle_sequence_id, Length: 26489, dtype: Int64

### Let's check min/max value of Int64

In [44]:
np.iinfo(np.int64)

iinfo(min=-9223372036854775808, max=9223372036854775807, dtype=int64)

In [45]:
df.event_bundle_sequence_id.max()

np.int64(9998038061)

In [46]:
df.event_bundle_sequence_id.min()

np.int64(-9999956402)

In [47]:
df.event_bundle_sequence_id.isna().sum()

np.int64(0)

In [48]:
df.event_server_timestamp_offset

0        <NA>
1        <NA>
2        <NA>
3        <NA>
4        <NA>
         ... 
26484    <NA>
26485    <NA>
26486    <NA>
26487    <NA>
26488    <NA>
Name: event_server_timestamp_offset, Length: 26489, dtype: Int64

In [49]:
df.event_server_timestamp_offset.isna().sum()

np.int64(26489)

In [50]:
df.user_id

0        None
1        None
2        None
3        None
4        None
         ... 
26484    None
26485    None
26486    None
26487    None
26488    None
Name: user_id, Length: 26489, dtype: object

In [51]:
df.user_id.isna().sum()

np.int64(26489)

In [54]:
df.user_pseudo_id

0           1026454.4271112504
1           1026454.4271112504
2           1026454.4271112504
3           1026454.4271112504
4           1026454.4271112504
                 ...          
26484    9021020864.5722086289
26485    9021020864.5722086289
26486    9021020864.5722086289
26487    9021020864.5722086289
26488    9050621716.2079146956
Name: user_pseudo_id, Length: 26489, dtype: object

In [55]:
df.user_pseudo_id.isna().sum()

np.int64(0)

In [56]:
df.user_first_touch_timestamp

0        1612069510766593
1        1612069510766593
2        1612069510766593
3        1612069510766593
4        1612069510766593
               ...       
26484    1611586297553897
26485    1611586297553897
26486    1611586297553897
26487    1611586297553897
26488    1612074944065877
Name: user_first_touch_timestamp, Length: 26489, dtype: Int64

In [57]:
df.user_first_touch_timestamp = pd.to_datetime(df.user_first_touch_timestamp, unit='us')

In [58]:
df.user_first_touch_timestamp

0       2021-01-31 05:05:10.766593
1       2021-01-31 05:05:10.766593
2       2021-01-31 05:05:10.766593
3       2021-01-31 05:05:10.766593
4       2021-01-31 05:05:10.766593
                   ...            
26484   2021-01-25 14:51:37.553897
26485   2021-01-25 14:51:37.553897
26486   2021-01-25 14:51:37.553897
26487   2021-01-25 14:51:37.553897
26488   2021-01-31 06:35:44.065877
Name: user_first_touch_timestamp, Length: 26489, dtype: datetime64[us]

In [59]:
df.user_first_touch_timestamp.isna().sum()

np.int64(958)

### The blank value for df.user_first_touch_timestamp should not be augmented, because it reflects none first time visit

In [60]:
df.app_info

0        None
1        None
2        None
3        None
4        None
         ... 
26484    None
26485    None
26486    None
26487    None
26488    None
Name: app_info, Length: 26489, dtype: object

In [61]:
df.app_info.isna().sum()

np.int64(26489)

In [11]:
df.stream_id

0        2100450278
1        2100450278
2        2100450278
3        2100450278
4        2100450278
            ...    
26484    2100450278
26485    2100450278
26486    2100450278
26487    2100450278
26488    2100450278
Name: stream_id, Length: 26489, dtype: Int64

In [12]:
df.stream_id.isna().sum()

np.int64(0)

In [13]:
df.platform.unique()

array(['WEB'], dtype=object)

In [14]:
df.event_dimensions.unique()

array([None], dtype=object)

# Nested Data

# From this table, there are 2 types of nested data:
* Fixed key-value dictionaries
* Array of key-value

In [36]:
from IPython.display import JSON

In [39]:
df.event_params[0].tolist()

[{'key': 'gclid', 'value': {'string_value': None, 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'gclsrc', 'value': {'string_value': None, 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'debug_mode', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_number', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'all_data', 'value': {'string_value': None, 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'entrances', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'session_engaged', 'value': {'string_value': '0', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_id',

In [1]:
df.event_params[0]

NameError: name 'df' is not defined

In [38]:
JSON(df.event_params[0].tolist(), expanded=True)

<IPython.core.display.JSON object>

### Let's see data dictionary for the user_properties
<b>https://support.google.com/analytics/answer/7029846?hl=en#zippy=%2Cuser</b>

In [102]:
df.user_properties

0        []
1        []
2        []
3        []
4        []
         ..
26484    []
26485    []
26486    []
26487    []
26488    []
Name: user_properties, Length: 26489, dtype: object

In [101]:
df[df.user_properties.map(len) > 0]

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items


In [34]:
JSON(df[df['items'].map(len) > 0]['items'][23530].tolist(), expanded=True)

<IPython.core.display.JSON object>

# Let's see it in a big picture and start to process it

In [42]:
pd.set_option('display.max_colwidth', None)

In [43]:
df[df.index==23530]

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
23530,20210131,1612091514175516,view_item,"[{'key': 'engaged_session_event', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'debug_mode', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'engagement_time_msec', 'value': {'string_value': None, 'int_value': 7344, 'float_value': None, 'double_value': None}}, {'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/asearch.html', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_id', 'value': {'string_value': None, 'int_value': 5349214666, 'float_value': None, 'double_value': None}}, {'key': 'page_title', 'value': {'string_value': 'Store search results', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'session_engaged', 'value': {'string_value': '1', 'int_value': None, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_number', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'page_referrer', 'value': {'string_value': None, 'int_value': None, 'float_value': None, 'double_value': None}}]",<NA>,NaN,-5814762287,<NA>,None,1540124.2144280285,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1612091231357744,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Apple', 'mobile_model_name': 'Safari', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': None, 'operating_system': 'Web', 'operating_system_version': 'Intel 10.15', 'vendor_id': None, 'advertising_id': None, 'language': None, 'is_limited_ad_tracking': 'No', 'time_zone_offset_seconds': None, 'web_info': {'browser': 'Chrome', 'browser_version': '86.0'}}","{'continent': 'Americas', 'sub_continent': 'Northern America', 'country': 'United States', 'region': 'Oregon', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'google'}",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_in_usd': None, 'shipping_value': None, 'tax_value_in_usd': None, 'tax_value': None, 'unique_items': 1, 'transaction_id': '(not set)'}","[{'item_id': 'GGOEAFBA115599', 'item_name': 'Android Super Hero 3D Framed Art', 'item_brand': 'Android', 'item_variant': '(not set)', 'item_category': '', 'item_category2': '(not set)', 'item_category3': '(not set)', 'item_category4': '(not set)', 'item_category5': '(not set)', 'price_in_usd': None, 'price': 40.0, 'quantity': None, 'item_revenue_in_usd': None, 'item_revenue': None, 'item_refund_in_usd': None, 'item_refund': None, 'coupon': '(not set)', 'affiliation': '(not set)', 'location_id': '(not set)', 'item_list_id': '(not set)', 'item_list_name': '(not set)', 'item_list_index': '4', 'promotion_id': '(not set)', 'promotion_name': '(not set)', 'creative_name': '(not set)', 'creative_slot': '(not set)'}]"
